In [ ]:
from pylab import *
from scipy.stats import linregress as linregress
import os
from scipy.interpolate import InterpolatedUnivariateSpline

# Write Data

In [ ]:
dirnames = [x[0] for x in os.walk(".")][2:]

In [ ]:
dirnames

In [ ]:
l = np.zeros(len(dirnames), dtype=int)
N = np.zeros(len(dirnames), dtype=int)
w = np.zeros(len(dirnames))
dr = np.zeros(len(dirnames))
M_Komar1 = np.zeros(len(dirnames))
M_Komar2 = np.zeros(len(dirnames))
J_Komar1 = np.zeros(len(dirnames))
J_Komar2 = np.zeros(len(dirnames))
phi_max = np.zeros(len(dirnames))
rr_max = np.zeros(len(dirnames))
min_alpha = np.zeros(len(dirnames))
min_beta = np.zeros(len(dirnames))
max_metric = np.zeros(len(dirnames))
max_a_metric = np.zeros(len(dirnames))
max_psi = np.zeros(len(dirnames))
min_lambda = np.zeros(len(dirnames))
r99 = np.zeros(len(dirnames))
ergoregion = np.zeros(len(dirnames), dtype=int)

In [ ]:
ghost = 2

In [ ]:
for i in range(len(dirnames)):
    # Get dirname.
    dirname = dirnames[i]
    
    # Get parameters.
    l[i], dr[i], N[i] = int(dirnames[i][4]), float64(dirnames[i][23:34]), int(dirnames[i][-4:])
    #print("Processing file l = %d, w = %1.5E, dr = %1.5E, N =%04d" % (l[i], w[i], dr[i], N[i]))

    # Axis.
    r_l = np.linspace(dr[i] * (0.5 - ghost), dr[i] * (N[i] + ghost - 0.5), N[i] + 2 * ghost)
    
    # Sanity checks.
    #if not(os.path.isfile(dirname + "/M_Komar1.asc")):
    #    print("Warning M_Komar1.asc does not exist in %d, %s!" % (i, dirname))
    #    continue
    #elif not(os.path.isfile(dirname + "/M_Komar2.asc")):
    #    print("Warning M_Komar2.asc does not exist in %d, %s!" % (i, dirname))
    #    continue
    #elif not(os.path.isfile(dirname + "/J_Komar1.asc")):
    #    print("Warning J_Komar2.asc does not exist in %d, %s!" % (i, dirname))
    #    continue
    #elif not(os.path.isfile(dirname + "/J_Komar2.asc")):
    #    print("Warning J_Komar2.asc does not exist in %d, %s!" % (i, dirname))
    #    continue
    #elif not(os.path.isfile(dirname + "/psi_f.asc")):
    #    print("Warning psi_f.asc does not exist in %d, %s!" % (i, dirname))
    #    continue

    # Get frequency, masses, and angular momenta.
    w[i] = np.genfromtxt(dirname + "/w_f.asc").item()
    M_Komar1[i] = np.genfromtxt(dirname + "/M_Komar1.asc", skip_header = N[i]).item()
    M_Komar2[i] = np.genfromtxt(dirname + "/M_Komar2.asc", skip_header = N[i]).item()
    J_Komar1[i] = np.genfromtxt(dirname + "/J_Komar1.asc", skip_header = N[i]).item()
    J_Komar2[i] = np.genfromtxt(dirname + "/J_Komar2.asc", skip_header = N[i]).item()

    # Import psi to get phi's maximum value and coordinate.
    min_alpha[i] = np.min(np.exp(np.genfromtxt(dirname + "/log_alpha_f.asc", usecols = ghost)))
    min_beta[i]  = np.min(np.genfromtxt(dirname + "/beta_f.asc", usecols = ghost))
    max_metric[i]   = np.max(np.exp(2.0 * np.genfromtxt(dirname + "/log_a_f.asc", usecols = ghost)))
    max_a_metric[i] = np.max(np.exp(2.0 * np.genfromtxt(dirname + "/log_h_f.asc", usecols = ghost)))
    max_psi[i] = np.genfromtxt(dirname + "/sph_psi_f.asc", usecols = 0, max_rows = 1)
    min_lambda[i] = np.min(np.genfromtxt(dirname + "/lambda_f.asc", usecols = ghost))
    
    phi_max[i] = np.genfromtxt(dirname + "/phi_max.asc", usecols = 0, max_rows = 1)
    rr_max[i] = np.genfromtxt(dirname + "/rr_phi_max.asc", usecols = 0, max_rows = 1)
    r99[i] = np.genfromtxt(dirname + "/r99.asc", usecols = 0, max_rows = 1)
    ergoregion[i] = np.genfromtxt(dirname + "/ergoregion_flag.asc", usecols = 0, max_rows = 1)

In [ ]:
l = l[max_psi.argsort()]
N = N[max_psi.argsort()]
w = w[max_psi.argsort()]
dr = dr[max_psi.argsort()]
M_Komar1 = M_Komar1[max_psi.argsort()]
M_Komar2 = M_Komar2[max_psi.argsort()]
J_Komar1 = J_Komar1[max_psi.argsort()]
J_Komar2 = J_Komar2[max_psi.argsort()]
phi_max = phi_max[max_psi.argsort()]
rr_max = rr_max[max_psi.argsort()]
r99 = r99[max_psi.argsort()]
ergoregion = ergoregion[max_psi.argsort()]
min_alpha = min_alpha[max_psi.argsort()]
min_beta = min_beta[max_psi.argsort()]
max_metric = max_metric[max_psi.argsort()]
max_a_metric = max_a_metric[max_psi.argsort()]
min_lambda = min_lambda[max_psi.argsort()]

In [ ]:
k_max = rr_max / dr

In [ ]:
array_dirnames = np.array(dirnames)
array_dirnames = array_dirnames[max_psi.argsort()]

In [ ]:
from shutil import move

In [ ]:
#for dirname in array_dirnames[k_max > 10.0]:
#    move(dirname, "../Catalogue3/" + dirname[2:])

In [ ]:
max_psi = max_psi[max_psi.argsort()]

In [ ]:
print(max(np.max(M_Komar1[(l == 1) * (N == 400) * (dr == 0.02)]), np.max(M_Komar1[(l == 1) * (N == 400) * (dr == 0.04)][:-11]), np.max(M_Komar1[(l == 1) * (N == 400) * (dr == 0.08)][:-11])))
print(max(np.max(M_Komar1[(l == 2) * (N == 400) * (dr == 0.02)]), np.max(M_Komar1[(l == 2) * (N == 400) * (dr == 0.04)][:-34]), np.max(M_Komar1[(l == 2) * (N == 400) * (dr == 0.08)][:-35])))
print(max(np.max(M_Komar1[(l == 3) * (N == 400) * (dr == 0.04)]), np.max(M_Komar1[(l == 3) * (N == 400) * (dr == 0.08)][:-18])))
print(max(np.max(M_Komar1[(l == 4) * (N == 400) * (dr == 0.04)]), np.max(M_Komar1[(l == 4) * (N == 400) * (dr == 0.08)][:-14])))
print(max(np.max(M_Komar1[(l == 5) * (N == 800) * (dr == 0.04)]), np.max(M_Komar1[(l == 5) * (N == 400) * (dr == 0.16)])))
print(max(np.max(M_Komar1[(l == 6) * (N == 800) * (dr == 0.04)]), np.max(M_Komar1[(l == 6) * (N == 400) * (dr == 0.16)])))

In [ ]:
print(max(np.max(J_Komar1[(l == 1) * (N == 400) * (dr == 0.02)]), np.max(J_Komar1[(l == 1) * (N == 400) * (dr == 0.04)][:-11]), np.max(J_Komar1[(l == 1) * (N == 400) * (dr == 0.08)][:-11])))
print(max(np.max(J_Komar1[(l == 2) * (N == 400) * (dr == 0.02)]), np.max(J_Komar1[(l == 2) * (N == 400) * (dr == 0.04)][:-34]), np.max(J_Komar1[(l == 2) * (N == 400) * (dr == 0.08)][:-35])))
print(max(np.max(J_Komar1[(l == 3) * (N == 400) * (dr == 0.04)]), np.max(J_Komar1[(l == 3) * (N == 400) * (dr == 0.08)][:-18])))
print(max(np.max(J_Komar1[(l == 4) * (N == 400) * (dr == 0.04)]), np.max(J_Komar1[(l == 4) * (N == 400) * (dr == 0.08)][:-14])))
print(max(np.max(J_Komar1[(l == 5) * (N == 800) * (dr == 0.04)]), np.max(J_Komar1[(l == 5) * (N == 400) * (dr == 0.16)])))
print(max(np.max(J_Komar1[(l == 6) * (N == 800) * (dr == 0.04)]), np.max(J_Komar1[(l == 6) * (N == 400) * (dr == 0.16)])))

In [ ]:
print(min(np.min(w[(l == 1) * (N == 400) * (dr == 0.02)]), np.min(w[(l == 1) * (N == 400) * (dr == 0.04)][:-11]), np.min(w[(l == 1) * (N == 400) * (dr == 0.08)][:-11])))
print(min(np.min(w[(l == 2) * (N == 400) * (dr == 0.02)]), np.min(w[(l == 2) * (N == 400) * (dr == 0.04)][:-34]), np.min(w[(l == 2) * (N == 400) * (dr == 0.08)][:-35])))
print(min(np.min(w[(l == 3) * (N == 400) * (dr == 0.04)]), np.min(w[(l == 3) * (N == 400) * (dr == 0.08)][:-18])))
print(min(np.min(w[(l == 4) * (N == 400) * (dr == 0.04)]), np.min(w[(l == 4) * (N == 400) * (dr == 0.08)][:-14])))
print(min(np.min(w[(l == 5) * (N == 800) * (dr == 0.04)]), np.min(w[(l == 5) * (N == 400) * (dr == 0.16)])))
print(min(np.min(w[(l == 6) * (N == 800) * (dr == 0.04)]), np.min(w[(l == 6) * (N == 400) * (dr == 0.16)])))

l = 1

In [ ]:
x0 = max_psi[(l == 1) * (N == 400) * (dr == 0.02)]
x1 = max_psi[(l == 1) * (N == 400) * (dr == 0.04)]
x2 = max_psi[(l == 1) * (N == 400) * (dr == 0.08)]

y0 = M_Komar1[(l == 1) * (N == 400) * (dr == 0.02)]
y1 = M_Komar1[(l == 1) * (N == 400) * (dr == 0.04)]
y2 = M_Komar1[(l == 1) * (N == 400) * (dr == 0.08)]

z0 = J_Komar1[(l == 1) * (N == 400) * (dr == 0.02)]
z1 = J_Komar1[(l == 1) * (N == 400) * (dr == 0.04)]
z2 = J_Komar1[(l == 1) * (N == 400) * (dr == 0.08)]

w0 = w[(l == 1) * (N == 400) * (dr == 0.02)]
w1 = w[(l == 1) * (N == 400) * (dr == 0.04)]
w2 = w[(l == 1) * (N == 400) * (dr == 0.08)]

u0 = phi_max[(l == 1) * (N == 400) * (dr == 0.02)]
u1 = phi_max[(l == 1) * (N == 400) * (dr == 0.04)]
u2 = phi_max[(l == 1) * (N == 400) * (dr == 0.08)]

v0 = rr_max[(l == 1) * (N == 400) * (dr == 0.02)]
v1 = rr_max[(l == 1) * (N == 400) * (dr == 0.04)]
v2 = rr_max[(l == 1) * (N == 400) * (dr == 0.08)]

In [ ]:
spline_y0 = InterpolatedUnivariateSpline(x0, y0)
spline_y1 = InterpolatedUnivariateSpline(x1, y1)
spline_y2 = InterpolatedUnivariateSpline(x2, y2)

spline_z0 = InterpolatedUnivariateSpline(x0, y0)
spline_z1 = InterpolatedUnivariateSpline(x1, y1)
spline_z2 = InterpolatedUnivariateSpline(x2, y2)

spline_w0 = InterpolatedUnivariateSpline(x0, w0)
spline_w1 = InterpolatedUnivariateSpline(x1, w1)
spline_w2 = InterpolatedUnivariateSpline(x2, w2)

spline_u0 = InterpolatedUnivariateSpline(x0, u0)
spline_u1 = InterpolatedUnivariateSpline(x1, u1)
spline_u2 = InterpolatedUnivariateSpline(x2, u2)

spline_v0 = InterpolatedUnivariateSpline(x0, v0)
spline_v1 = InterpolatedUnivariateSpline(x1, v1)
spline_v2 = InterpolatedUnivariateSpline(x2, v2)

In [ ]:
dx01 = np.linspace(np.min(x0), np.max(x1), 101)
dx12 = np.linspace(np.min(x1), np.max(x2), 101)

In [ ]:
s01 = dx01[np.argmin(np.abs(spline_y0(dx01) - spline_y1(dx01)))]

In [ ]:
s02 = dx12[np.argmin(np.abs(spline_y1(dx12) - spline_y2(dx12)))]

In [ ]:
print(s01, s02)

In [ ]:
dat_x = np.concatenate((x2[x2 < s02], x1[x1 < s01][s02 < x1[x1 < s01]], x0[x0 > s01]))
dat_y = np.concatenate((y2[x2 < s02], y1[x1 < s01][s02 < x1[x1 < s01]], y0[x0 > s01]))
dat_z = np.concatenate((z2[x2 < s02], z1[x1 < s01][s02 < x1[x1 < s01]], z0[x0 > s01]))
dat_w = np.concatenate((w2[x2 < s02], w1[x1 < s01][s02 < x1[x1 < s01]], w0[x0 > s01]))
dat_u = np.concatenate((u2[x2 < s02], u1[x1 < s01][s02 < x1[x1 < s01]], u0[x0 > s01]))
dat_v = np.concatenate((v2[x2 < s02], v1[x1 < s01][s02 < x1[x1 < s01]], v0[x0 > s01]))

In [ ]:
l1 = np.stack((dat_x, dat_y, dat_z, dat_w, dat_u, dat_v))

In [ ]:
np.savetxt("l=1.asc", l1.T, header = "Data for l=1\npsi(0) M_Komar J_Komar w max(phi) rr(max(phi))")

l = 2

In [ ]:
x0 = max_psi[(l == 2) * (N == 400) * (dr == 0.02)]
x1 = max_psi[(l == 2) * (N == 400) * (dr == 0.04)]
x2 = max_psi[(l == 2) * (N == 400) * (dr == 0.08)]

y0 = M_Komar1[(l == 2) * (N == 400) * (dr == 0.02)]
y1 = M_Komar1[(l == 2) * (N == 400) * (dr == 0.04)]
y2 = M_Komar1[(l == 2) * (N == 400) * (dr == 0.08)]

z0 = J_Komar1[(l == 2) * (N == 400) * (dr == 0.02)]
z1 = J_Komar1[(l == 2) * (N == 400) * (dr == 0.04)]
z2 = J_Komar1[(l == 2) * (N == 400) * (dr == 0.08)]

w0 = w[(l == 2) * (N == 400) * (dr == 0.02)]
w1 = w[(l == 2) * (N == 400) * (dr == 0.04)]
w2 = w[(l == 2) * (N == 400) * (dr == 0.08)]

u0 = phi_max[(l == 2) * (N == 400) * (dr == 0.02)]
u1 = phi_max[(l == 2) * (N == 400) * (dr == 0.04)]
u2 = phi_max[(l == 2) * (N == 400) * (dr == 0.08)]

v0 = rr_max[(l == 2) * (N == 400) * (dr == 0.02)]
v1 = rr_max[(l == 2) * (N == 400) * (dr == 0.04)]
v2 = rr_max[(l == 2) * (N == 400) * (dr == 0.08)]

In [ ]:
spline_y0 = InterpolatedUnivariateSpline(x0, y0)
spline_y1 = InterpolatedUnivariateSpline(x1, y1)
spline_y2 = InterpolatedUnivariateSpline(x2, y2)

spline_z0 = InterpolatedUnivariateSpline(x0, y0)
spline_z1 = InterpolatedUnivariateSpline(x1, y1)
spline_z2 = InterpolatedUnivariateSpline(x2, y2)

spline_w0 = InterpolatedUnivariateSpline(x0, w0)
spline_w1 = InterpolatedUnivariateSpline(x1, w1)
spline_w2 = InterpolatedUnivariateSpline(x2, w2)

spline_u0 = InterpolatedUnivariateSpline(x0, u0)
spline_u1 = InterpolatedUnivariateSpline(x1, u1)
spline_u2 = InterpolatedUnivariateSpline(x2, u2)

spline_v0 = InterpolatedUnivariateSpline(x0, v0)
spline_v1 = InterpolatedUnivariateSpline(x1, v1)
spline_v2 = InterpolatedUnivariateSpline(x2, v2)

In [ ]:
dx01 = np.linspace(np.min(x0), np.max(x1), 101)
dx12 = np.linspace(np.min(x1), np.max(x2), 101)

In [ ]:
s01 = dx01[np.argmin(np.abs(spline_y0(dx01) - spline_y1(dx01)))]

In [ ]:
s02 = dx12[np.argmin(np.abs(spline_y1(dx12) - spline_y2(dx12)))]

In [ ]:
print(s01, s02)

In [ ]:
dat_x = np.concatenate((x2[x2 < s02], x1[x1 < s01][s02 < x1[x1 < s01]], x0[x0 > s01]))
dat_y = np.concatenate((y2[x2 < s02], y1[x1 < s01][s02 < x1[x1 < s01]], y0[x0 > s01]))
dat_z = np.concatenate((z2[x2 < s02], z1[x1 < s01][s02 < x1[x1 < s01]], z0[x0 > s01]))
dat_w = np.concatenate((w2[x2 < s02], w1[x1 < s01][s02 < x1[x1 < s01]], w0[x0 > s01]))
dat_u = np.concatenate((u2[x2 < s02], u1[x1 < s01][s02 < x1[x1 < s01]], u0[x0 > s01]))
dat_v = np.concatenate((v2[x2 < s02], v1[x1 < s01][s02 < x1[x1 < s01]], v0[x0 > s01]))

In [ ]:
l2 = np.stack((dat_x, dat_y, dat_z, dat_w, dat_u, dat_v))

In [ ]:
np.savetxt("l=2.asc", l2.T, header = "Data for l=2\npsi(0) M_Komar J_Komar w max(phi) rr(max(phi))")

l = 3

In [ ]:
x1 = max_psi[(l == 3) * (N == 400) * (dr == 0.04)]
x2 = max_psi[(l == 3) * (N == 400) * (dr == 0.08)]

y1 = M_Komar1[(l == 3) * (N == 400) * (dr == 0.04)]
y2 = M_Komar1[(l == 3) * (N == 400) * (dr == 0.08)]

z1 = J_Komar1[(l == 3) * (N == 400) * (dr == 0.04)]
z2 = J_Komar1[(l == 3) * (N == 400) * (dr == 0.08)]

w1 = w[(l == 3) * (N == 400) * (dr == 0.04)]
w2 = w[(l == 3) * (N == 400) * (dr == 0.08)]

u1 = phi_max[(l == 3) * (N == 400) * (dr == 0.04)]
u2 = phi_max[(l == 3) * (N == 400) * (dr == 0.08)]

v1 = rr_max[(l == 3) * (N == 400) * (dr == 0.04)]
v2 = rr_max[(l == 3) * (N == 400) * (dr == 0.08)]

In [ ]:
spline_y1 = InterpolatedUnivariateSpline(x1, y1)
spline_y2 = InterpolatedUnivariateSpline(x2, y2)

spline_z1 = InterpolatedUnivariateSpline(x1, y1)
spline_z2 = InterpolatedUnivariateSpline(x2, y2)

spline_w1 = InterpolatedUnivariateSpline(x1, w1)
spline_w2 = InterpolatedUnivariateSpline(x2, w2)

spline_u1 = InterpolatedUnivariateSpline(x1, u1)
spline_u2 = InterpolatedUnivariateSpline(x2, u2)

spline_v1 = InterpolatedUnivariateSpline(x1, v1)
spline_v2 = InterpolatedUnivariateSpline(x2, v2)

In [ ]:
dx12 = np.linspace(np.min(x1), np.max(x2), 101)

In [ ]:
s02 = dx12[np.argmin(np.abs(spline_y1(dx12) - spline_y2(dx12)))]

In [ ]:
print(s02)

In [ ]:
dat_x = np.concatenate((x2[x2 < s02], x1[x1 > s02]))
dat_y = np.concatenate((y2[x2 < s02], y1[x1 > s02]))
dat_z = np.concatenate((z2[x2 < s02], z1[x1 > s02]))
dat_w = np.concatenate((w2[x2 < s02], w1[x1 > s02]))
dat_u = np.concatenate((u2[x2 < s02], u1[x1 > s02]))
dat_v = np.concatenate((v2[x2 < s02], v1[x1 > s02]))

In [ ]:
l3 = np.stack((dat_x, dat_y, dat_z, dat_w, dat_u, dat_v))

In [ ]:
np.savetxt("l=3.asc", l3.T, header = "Data for l=3\npsi(0) M_Komar J_Komar w max(phi) rr(max(phi))")

l = 4

In [ ]:
x1 = max_psi[(l == 4) * (N == 400) * (dr == 0.04)]
x2 = max_psi[(l == 4) * (N == 400) * (dr == 0.08)]

y1 = M_Komar1[(l == 4) * (N == 400) * (dr == 0.04)]
y2 = M_Komar1[(l == 4) * (N == 400) * (dr == 0.08)]

z1 = J_Komar1[(l == 4) * (N == 400) * (dr == 0.04)]
z2 = J_Komar1[(l == 4) * (N == 400) * (dr == 0.08)]

w1 = w[(l == 4) * (N == 400) * (dr == 0.04)]
w2 = w[(l == 4) * (N == 400) * (dr == 0.08)]

u1 = phi_max[(l == 4) * (N == 400) * (dr == 0.04)]
u2 = phi_max[(l == 4) * (N == 400) * (dr == 0.08)]

v1 = rr_max[(l == 4) * (N == 400) * (dr == 0.04)]
v2 = rr_max[(l == 4) * (N == 400) * (dr == 0.08)]

In [ ]:
spline_y1 = InterpolatedUnivariateSpline(x1, y1)
spline_y2 = InterpolatedUnivariateSpline(x2, y2)

spline_z1 = InterpolatedUnivariateSpline(x1, y1)
spline_z2 = InterpolatedUnivariateSpline(x2, y2)

spline_w1 = InterpolatedUnivariateSpline(x1, w1)
spline_w2 = InterpolatedUnivariateSpline(x2, w2)

spline_u1 = InterpolatedUnivariateSpline(x1, u1)
spline_u2 = InterpolatedUnivariateSpline(x2, u2)

spline_v1 = InterpolatedUnivariateSpline(x1, v1)
spline_v2 = InterpolatedUnivariateSpline(x2, v2)

In [ ]:
dx12 = np.linspace(np.min(x1), np.max(x2), 101)

In [ ]:
s02 = dx12[np.argmin(np.abs(spline_y1(dx12) - spline_y2(dx12)))]

In [ ]:
print(s02)

In [ ]:
dat_x = np.concatenate((x2[x2 < s02], x1[x1 > s02]))
dat_y = np.concatenate((y2[x2 < s02], y1[x1 > s02]))
dat_z = np.concatenate((z2[x2 < s02], z1[x1 > s02]))
dat_w = np.concatenate((w2[x2 < s02], w1[x1 > s02]))
dat_u = np.concatenate((u2[x2 < s02], u1[x1 > s02]))
dat_v = np.concatenate((v2[x2 < s02], v1[x1 > s02]))

In [ ]:
l4 = np.stack((dat_x, dat_y, dat_z, dat_w, dat_u, dat_v))

In [ ]:
np.savetxt("l=4.asc", l4.T, header = "Data for l=4\npsi(0) M_Komar J_Komar w max(phi) rr(max(phi))")

l = 5

In [ ]:
x1 = max_psi[(l == 5) * (N == 800) * (dr == 0.04)]
x2 = max_psi[(l == 5) * (N == 400) * (dr == 0.16)]

y1 = M_Komar1[(l == 5) * (N == 800) * (dr == 0.04)]
y2 = M_Komar1[(l == 5) * (N == 400) * (dr == 0.16)]

z1 = J_Komar1[(l == 5) * (N == 800) * (dr == 0.04)]
z2 = J_Komar1[(l == 5) * (N == 400) * (dr == 0.16)]

w1 = w[(l == 5) * (N == 800) * (dr == 0.04)]
w2 = w[(l == 5) * (N == 400) * (dr == 0.16)]

u1 = phi_max[(l == 5) * (N == 800) * (dr == 0.04)]
u2 = phi_max[(l == 5) * (N == 400) * (dr == 0.16)]

v1 = rr_max[(l == 5) * (N == 800) * (dr == 0.04)]
v2 = rr_max[(l == 5) * (N == 400) * (dr == 0.16)]

In [ ]:
spline_y1 = InterpolatedUnivariateSpline(x1, y1)
spline_y2 = InterpolatedUnivariateSpline(x2, y2)

spline_z1 = InterpolatedUnivariateSpline(x1, y1)
spline_z2 = InterpolatedUnivariateSpline(x2, y2)

spline_w1 = InterpolatedUnivariateSpline(x1, w1)
spline_w2 = InterpolatedUnivariateSpline(x2, w2)

spline_u1 = InterpolatedUnivariateSpline(x1, u1)
spline_u2 = InterpolatedUnivariateSpline(x2, u2)

spline_v1 = InterpolatedUnivariateSpline(x1, v1)
spline_v2 = InterpolatedUnivariateSpline(x2, v2)

In [ ]:
dx12 = np.linspace(np.min(x1), np.max(x2), 101)

In [ ]:
s02 = dx12[np.argmin(np.abs(spline_y1(dx12) - spline_y2(dx12)))]

In [ ]:
print(s02)

In [ ]:
dat_x = np.concatenate((x2[x2 < s02], x1[x1 > s02]))
dat_y = np.concatenate((y2[x2 < s02], y1[x1 > s02]))
dat_z = np.concatenate((z2[x2 < s02], z1[x1 > s02]))
dat_w = np.concatenate((w2[x2 < s02], w1[x1 > s02]))
dat_u = np.concatenate((u2[x2 < s02], u1[x1 > s02]))
dat_v = np.concatenate((v2[x2 < s02], v1[x1 > s02]))

In [ ]:
l5 = np.stack((dat_x, dat_y, dat_z, dat_w, dat_u, dat_v))

In [ ]:
np.savetxt("l=5.asc", l5.T, header = "Data for l=5\npsi(0) M_Komar J_Komar w max(phi) rr(max(phi))")

l = 6

In [ ]:
x1 = max_psi[(l == 6) * (N == 800) * (dr == 0.04)]
x2 = max_psi[(l == 6) * (N == 400) * (dr == 0.16)]

y1 = M_Komar1[(l == 6) * (N == 800) * (dr == 0.04)]
y2 = M_Komar1[(l == 6) * (N == 400) * (dr == 0.16)]

z1 = J_Komar1[(l == 6) * (N == 800) * (dr == 0.04)]
z2 = J_Komar1[(l == 6) * (N == 400) * (dr == 0.16)]

w1 = w[(l == 6) * (N == 800) * (dr == 0.04)]
w2 = w[(l == 6) * (N == 400) * (dr == 0.16)]

u1 = phi_max[(l == 6) * (N == 800) * (dr == 0.04)]
u2 = phi_max[(l == 6) * (N == 400) * (dr == 0.16)]

v1 = rr_max[(l == 6) * (N == 800) * (dr == 0.04)]
v2 = rr_max[(l == 6) * (N == 400) * (dr == 0.16)]

In [ ]:
spline_y1 = InterpolatedUnivariateSpline(x1, y1)
spline_y2 = InterpolatedUnivariateSpline(x2, y2)

spline_z1 = InterpolatedUnivariateSpline(x1, y1)
spline_z2 = InterpolatedUnivariateSpline(x2, y2)

spline_w1 = InterpolatedUnivariateSpline(x1, w1)
spline_w2 = InterpolatedUnivariateSpline(x2, w2)

spline_u1 = InterpolatedUnivariateSpline(x1, u1)
spline_u2 = InterpolatedUnivariateSpline(x2, u2)

spline_v1 = InterpolatedUnivariateSpline(x1, v1)
spline_v2 = InterpolatedUnivariateSpline(x2, v2)

In [ ]:
dx12 = np.linspace(np.min(x1), np.max(x2), 101)

In [ ]:
s02 = dx12[np.argmin(np.abs(spline_y1(dx12) - spline_y2(dx12)))]

In [ ]:
print(s02)

In [ ]:
dat_x = np.concatenate((x2[x2 < s02], x1[x1 > s02]))
dat_y = np.concatenate((y2[x2 < s02], y1[x1 > s02]))
dat_z = np.concatenate((z2[x2 < s02], z1[x1 > s02]))
dat_w = np.concatenate((w2[x2 < s02], w1[x1 > s02]))
dat_u = np.concatenate((u2[x2 < s02], u1[x1 > s02]))
dat_v = np.concatenate((v2[x2 < s02], v1[x1 > s02]))

In [ ]:
l6 = np.stack((dat_x, dat_y, dat_z, dat_w, dat_u, dat_v))

In [ ]:
np.savetxt("l=6.asc", l6.T, header = "Data for l=6\npsi(0) M_Komar J_Komar w max(phi) rr(max(phi))")

# Retrieve Data

In [ ]:
l0 = np.genfromtxt("../../Boson Stars/l=0.asc")
l1 = np.genfromtxt("../../Boson Stars/l=1.asc")
l2 = np.genfromtxt("../../Boson Stars/l=2.asc")
l3 = np.genfromtxt("../../Boson Stars/l=3.asc")
l4 = np.genfromtxt("../../Boson Stars/l=4.asc")
l5 = np.genfromtxt("../../Boson Stars/l=5.asc")
l6 = np.genfromtxt("../../Boson Stars/l=6.asc")

In [ ]:
boson = [l0, l1, l2, l3, l4, l5, l6]

In [ ]:
splines = [[],[],[],[],[],[],[]]
for i in range(7):
    for j in range(1,6):
        splines[i].append(InterpolatedUnivariateSpline(boson[i][:,0], boson[i][:,j], k = 4))

In [ ]:
cr_M_pts = [splines[i][0].derivative().roots() for i in range(7)]
cr_M_vals = [splines[i][0](cr_M_pts[i]) for i in range(7)]

In [ ]:
cr_J_pts = [splines[i][1].derivative().roots() for i in range(7)]
cr_J_vals = [splines[i][1](cr_J_pts[i]) for i in range(7)]

In [ ]:
cr_w_pts = [splines[i][2].derivative().roots() for i in range(7)]
cr_w_vals = [splines[i][2](cr_w_pts[i]) for i in range(7)]

In [ ]:
cr_M_vals

In [ ]:
cr_J_vals

In [ ]:
cr_w_vals

# Draw Figures

In [ ]:
plt.rcParams.update({'font.size': 12})

In [ ]:
fig, ax = plt.subplots(figsize = (16, 12), nrows=2, ncols=2)

plt.suptitle(r"Characterization for Boson Stars $l \in [0,\,6]$")

ax[0][0].set_title("Mass")

ax[0][0].plot(boson[0][:,3], boson[0][:,1], 'b.', markersize = 2)
ax[0][0].plot(boson[1][:,3], boson[1][:,1], 'g.', markersize = 2)
ax[0][0].plot(boson[2][:,3], boson[2][:,1], 'r.', markersize = 2)
ax[0][0].plot(boson[3][:,3], boson[3][:,1], 'c.', markersize = 2)
ax[0][0].plot(boson[4][:,3], boson[4][:,1], 'm.', markersize = 2)
ax[0][0].plot(boson[5][:,3], boson[5][:,1], 'y.', markersize = 2)
ax[0][0].plot(boson[6][:,3], boson[6][:,1], 'k.', markersize = 2)

ax[0][0].plot(splines[0][2](boson[0][:,0]), splines[0][0](boson[0][:,0]), 'b-', label = r"$l = 0$")
ax[0][0].plot(splines[1][2](boson[1][:,0]), splines[1][0](boson[1][:,0]), 'g-', label = r"$l = 1$")
ax[0][0].plot(splines[2][2](boson[2][:,0]), splines[2][0](boson[2][:,0]), 'r-', label = r"$l = 2$")
ax[0][0].plot(splines[3][2](boson[3][:,0]), splines[3][0](boson[3][:,0]), 'c-', label = r"$l = 3$")
ax[0][0].plot(splines[4][2](boson[4][:,0]), splines[4][0](boson[4][:,0]), 'm-', label = r"$l = 4$")
ax[0][0].plot(splines[5][2](boson[5][:,0]), splines[5][0](boson[5][:,0]), 'y-', label = r"$l = 5$")
ax[0][0].plot(splines[6][2](boson[6][:,0]), splines[6][0](boson[6][:,0]), 'k-', label = r"$l = 6$")

ax[0][0].legend()
ax[0][0].set_xlabel(r"$\omega\;[m/\hbar]$")
ax[0][0].set_ylabel(r"$M\;[m^2_P/m]$")
#ax[0][0].set_ylim(0, 5)
#ax[0][0].set_xlim(0.5, 1.0)

ax[0][1].set_title("Angular Momentum")

ax[0][1].plot(boson[1][:,3], boson[1][:,2], 'g.', markersize = 3)
ax[0][1].plot(boson[2][:,3], boson[2][:,2], 'r.', markersize = 3)
ax[0][1].plot(boson[3][:,3], boson[3][:,2], 'c.', markersize = 3)
ax[0][1].plot(boson[4][:,3], boson[4][:,2], 'm.', markersize = 3)
ax[0][1].plot(boson[5][:,3], boson[5][:,2], 'y.', markersize = 3)
ax[0][1].plot(boson[6][:,3], boson[6][:,2], 'k.', markersize = 3)

ax[0][1].plot(splines[1][2](boson[1][:,0]), splines[1][1](boson[1][:,0]), 'g-', label = r"$l = 1$")
ax[0][1].plot(splines[2][2](boson[2][:,0]), splines[2][1](boson[2][:,0]), 'r-', label = r"$l = 2$")
ax[0][1].plot(splines[3][2](boson[3][:,0]), splines[3][1](boson[3][:,0]), 'c-', label = r"$l = 3$")
ax[0][1].plot(splines[4][2](boson[4][:,0]), splines[4][1](boson[4][:,0]), 'm-', label = r"$l = 4$")
ax[0][1].plot(splines[5][2](boson[5][:,0]), splines[5][1](boson[5][:,0]), 'y-', label = r"$l = 5$")
ax[0][1].plot(splines[6][2](boson[6][:,0]), splines[6][1](boson[6][:,0]), 'k-', label = r"$l = 6$")

ax[0][1].legend()
ax[0][1].set_xlabel(r"$\omega\;[m/\hbar]$")
ax[0][1].set_ylabel(r"$J\;[(m^2_P/m)^2]$")
#ax[0][1].set_ylim(0, 22)
#ax[0][1].set_xlim(0.5, 1.0)

ax[1][0].set_title("Scalar Field Maximum")

ax[1][0].plot(boson[0][:,4], boson[0][:,3], 'b.', markersize = 3)
ax[1][0].plot(boson[1][:,4], boson[1][:,3], 'g.', markersize = 3)
ax[1][0].plot(boson[2][:,4], boson[2][:,3], 'r.', markersize = 3)
ax[1][0].plot(boson[3][:,4], boson[3][:,3], 'c.', markersize = 3)
ax[1][0].plot(boson[4][:,4], boson[4][:,3], 'm.', markersize = 3)
ax[1][0].plot(boson[5][:,4], boson[5][:,3], 'y.', markersize = 3)
ax[1][0].plot(boson[6][:,4], boson[6][:,3], 'k.', markersize = 3)

ax[1][0].plot(splines[0][3](boson[0][:,0]), splines[0][2](boson[0][:,0]), 'b-', label = r"$l = 0$")
ax[1][0].plot(splines[1][3](boson[1][:,0]), splines[1][2](boson[1][:,0]), 'g-', label = r"$l = 1$")
ax[1][0].plot(splines[2][3](boson[2][:,0]), splines[2][2](boson[2][:,0]), 'r-', label = r"$l = 2$")
ax[1][0].plot(splines[3][3](boson[3][:,0]), splines[3][2](boson[3][:,0]), 'c-', label = r"$l = 3$")
ax[1][0].plot(splines[4][3](boson[4][:,0]), splines[4][2](boson[4][:,0]), 'm-', label = r"$l = 4$")
ax[1][0].plot(splines[5][3](boson[5][:,0]), splines[5][2](boson[5][:,0]), 'y-', label = r"$l = 5$")
ax[1][0].plot(splines[6][3](boson[6][:,0]), splines[6][2](boson[6][:,0]), 'k-', label = r"$l = 6$")

ax[1][0].legend()
ax[1][0].set_xlabel(r"$\max(\phi)$")
ax[1][0].set_ylabel(r"$\omega\;[m/\hbar]$")
#ax[1][0].set_ylim(0.5, 1.0)
ax[1][0].set_xlim(0, 0.2)

ax[1][1].set_title("Scalar Field Maximum Radial Coordinate")

ax[1][1].plot(boson[1][:,5], boson[1][:,1], 'g.', markersize = 3)
ax[1][1].plot(boson[2][:,5], boson[2][:,1], 'r.', markersize = 3)
ax[1][1].plot(boson[3][:,5], boson[3][:,1], 'c.', markersize = 3)
ax[1][1].plot(boson[4][:,5], boson[4][:,1], 'm.', markersize = 3)
ax[1][1].plot(boson[5][:,5], boson[5][:,1], 'y.', markersize = 3)
ax[1][1].plot(boson[6][:,5], boson[6][:,1], 'k.', markersize = 3)

ax[1][1].plot(splines[1][4](boson[1][:,0]), splines[1][0](boson[1][:,0]), 'g-', label = r"$l = 1$")
ax[1][1].plot(splines[2][4](boson[2][:,0]), splines[2][0](boson[2][:,0]), 'r-', label = r"$l = 2$")
ax[1][1].plot(splines[3][4](boson[3][:,0]), splines[3][0](boson[3][:,0]), 'c-', label = r"$l = 3$")
ax[1][1].plot(splines[4][4](boson[4][:,0]), splines[4][0](boson[4][:,0]), 'm-', label = r"$l = 4$")
ax[1][1].plot(splines[5][4](boson[5][:,0]), splines[5][0](boson[5][:,0]), 'y-', label = r"$l = 5$")
ax[1][1].plot(splines[6][4](boson[6][:,0]), splines[6][0](boson[6][:,0]), 'k-', label = r"$l = 6$")

ax[1][1].legend()
ax[1][1].set_xlabel(r"$r-$coordinate of $\max(\phi)\;[\hbar/m]$")
ax[1][1].set_ylabel(r"$M\;[m^2_P/m]$")
#ax[1][1].set_ylim(0, 5)
#ax[1][1].set_xlim(0, 25)

plt.show()

In [ ]:
np.min(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75]), np.max(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75])

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
fig, ax = plt.subplots(figsize = (16, 18), nrows=3, ncols=2)

plt.suptitle(r"Characterization for Rotating Boson Star $l = 6$")

ax[0][0].set_title(r"Lapse $\alpha$ Minimum Value")
ax[0][0].plot(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75], min_alpha[l == 6][dr[l == 6] == 4.0E-02][:-75], "b.-", label = r"$\Delta \rho = 4/100$")

ax[0][0].set_xlabel(r"$\psi_0\,[(\hbar/m)^6]$")
ax[0][0].set_ylabel(r"$\min(\alpha)$")

ax[0][1].set_title(r"Shift $\Omega$ Minimum Value")
ax[0][1].plot(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75], min_beta[l == 6][dr[l == 6] == 4.0E-02][:-75], "b.-", label = r"$\Delta \rho = 4/100$")

ax[0][1].set_xlabel(r"$\psi_0\,[(\hbar/m)^6]$")
ax[0][1].set_ylabel(r"$\min(\Omega)$")

ax[1][0].set_title(r"Metric $A = \gamma_{rr}$ Maximum Value")
ax[1][0].plot(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75], max_metric[l == 6][dr[l == 6] == 4.0E-02][:-75], "b.-", label = r"$\Delta \rho = 4/100$")

ax[1][0].set_xlabel(r"$\psi_0\,[(\hbar/m)^6]$")
ax[1][0].set_ylabel(r"$\max(A)$")

ax[1][1].set_title(r"Metric $H = \gamma_{\varphi \varphi} / \rho^2$ Maximum Value")
ax[1][1].plot(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75], max_a_metric[l == 6][dr[l == 6] == 4.0E-02][:-75], "b.-", label = r"$\Delta \rho = 4/100$")

ax[1][1].set_xlabel(r"$\psi_0\,[(\hbar/m)^6]$")
ax[1][1].set_ylabel(r"$\max(H)$")

ax[2][0].set_title(r"Scalar Field Frequency $\omega$")
ax[2][0].plot(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75], w[l == 6][dr[l == 6] == 4.0E-02][:-75], "b.-", label = r"$\omega = 4/100$")

ax[2][0].set_xlabel(r"$\psi_0\,[(\hbar/m)^6]$")
ax[2][0].set_ylabel(r"$\omega\,[m/\hbar]$")

ax[2][1].set_title(r"Regularization Variable $\lambda$ Minimum Value")
ax[2][1].plot(max_psi[l == 6][dr[l == 6] == 4.0E-02][:-75], min_lambda[l == 6][dr[l == 6] == 4.0E-02][:-75], "b.-", label = r"$\Delta \rho = 4/100$")

ax[2][1].set_xlabel(r"$\psi_0\,[(\hbar/m)^6]$")
ax[2][1].set_ylabel(r"$\min(\lambda)$")

fig.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

In [ ]:
np.min(max_psi[l == 1][dr[l == 1] == 8.0E-02]), np.max(max_psi[l == 1][dr[l == 1] == 2.0E-02])

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
fig, ax = plt.subplots(figsize = (16, 18), nrows=3, ncols=2)

plt.suptitle(r"Characterization for Rotating Boson Star $l = 1$")

ax[0][0].set_title(r"Lapse $\alpha$ Minimum Value")
ax[0][0].plot(max_psi[l == 1][dr[l == 1] == 2.0E-02][:], min_alpha[l == 1][dr[l == 1] == 2.0E-02][:], "b.-", label = r"$\Delta \rho = 2/100$")
ax[0][0].plot(max_psi[l == 1][dr[l == 1] == 4.0E-02][:], min_alpha[l == 1][dr[l == 1] == 4.0E-02][:], "b.-", label = r"$\Delta \rho = 4/100$")
ax[0][0].plot(max_psi[l == 1][dr[l == 1] == 8.0E-02][:], min_alpha[l == 1][dr[l == 1] == 8.0E-02][:], "b.-", label = r"$\Delta \rho = 8/100$")

ax[0][0].set_xlabel(r"$\psi_0\,[\hbar/m]$")
ax[0][0].set_ylabel(r"$\min(\alpha)$")

ax[0][1].set_title(r"Shift $\Omega$ Minimum Value")
ax[0][1].plot(max_psi[l == 1][dr[l == 1] == 2.0E-02][:], min_beta[l == 1][dr[l == 1] == 2.0E-02][:], "b.-", label = r"$\Delta \rho = 2/100$")
ax[0][1].plot(max_psi[l == 1][dr[l == 1] == 4.0E-02][:], min_beta[l == 1][dr[l == 1] == 4.0E-02][:], "b.-", label = r"$\Delta \rho = 4/100$")
ax[0][1].plot(max_psi[l == 1][dr[l == 1] == 8.0E-02][:], min_beta[l == 1][dr[l == 1] == 8.0E-02][:], "b.-", label = r"$\Delta \rho = 8/100$")

ax[0][1].set_xlabel(r"$\psi_0\,[\hbar/m]$")
ax[0][1].set_ylabel(r"$\min(\Omega)$")

ax[1][0].set_title(r"Metric $A = \gamma_{rr}$ Maximum Value")
ax[1][0].plot(max_psi[l == 1][dr[l == 1] == 2.0E-02][:], max_metric[l == 1][dr[l == 1] == 2.0E-02][:], "b.-", label = r"$\Delta \rho = 2/100$")
ax[1][0].plot(max_psi[l == 1][dr[l == 1] == 4.0E-02][:], max_metric[l == 1][dr[l == 1] == 4.0E-02][:], "b.-", label = r"$\Delta \rho = 4/100$")
ax[1][0].plot(max_psi[l == 1][dr[l == 1] == 8.0E-02][:], max_metric[l == 1][dr[l == 1] == 8.0E-02][:], "b.-", label = r"$\Delta \rho = 8/100$")

ax[1][0].set_xlabel(r"$\psi_0\,[\hbar/m]$")
ax[1][0].set_ylabel(r"$\max(A)$")

ax[1][1].set_title(r"Metric $H = \gamma_{\varphi \varphi} / \rho^2$ Maximum Value")
ax[1][1].plot(max_psi[l == 1][dr[l == 1] == 2.0E-02][:], max_a_metric[l == 1][dr[l == 1] == 2.0E-02][:], "b.-", label = r"$\Delta \rho = 2/100$")
ax[1][1].plot(max_psi[l == 1][dr[l == 1] == 4.0E-02][:], max_a_metric[l == 1][dr[l == 1] == 4.0E-02][:], "b.-", label = r"$\Delta \rho = 4/100$")
ax[1][1].plot(max_psi[l == 1][dr[l == 1] == 8.0E-02][:], max_a_metric[l == 1][dr[l == 1] == 8.0E-02][:], "b.-", label = r"$\Delta \rho = 8/100$")

ax[1][1].set_xlabel(r"$\psi_0\,[\hbar/m]$")
ax[1][1].set_ylabel(r"$\max(H)$")

ax[2][0].set_title(r"Scalar Field Frequency $\omega$")
ax[2][0].plot(max_psi[l == 1][dr[l == 1] == 2.0E-02][:], w[l == 1][dr[l == 1] == 2.0E-02][:], "b.-", label = r"$\omega = 2/100$")
ax[2][0].plot(max_psi[l == 1][dr[l == 1] == 4.0E-02][:], w[l == 1][dr[l == 1] == 4.0E-02][:], "b.-", label = r"$\omega = 4/100$")
ax[2][0].plot(max_psi[l == 1][dr[l == 1] == 8.0E-02][:], w[l == 1][dr[l == 1] == 8.0E-02][:], "b.-", label = r"$\omega = 8/100$")

ax[2][0].set_xlabel(r"$\psi_0\,[\hbar/m]$")
ax[2][0].set_ylabel(r"$\omega\,[m/\hbar]$")

ax[2][1].set_title(r"Regularization Variable $\lambda$ Minimum Value")
ax[2][1].plot(max_psi[l == 1][dr[l == 1] == 2.0E-02][:], min_lambda[l == 1][dr[l == 1] == 2.0E-02][:], "b.-", label = r"$\Delta \rho = 2/100$")
ax[2][1].plot(max_psi[l == 1][dr[l == 1] == 4.0E-02][:], min_lambda[l == 1][dr[l == 1] == 4.0E-02][:], "b.-", label = r"$\Delta \rho = 4/100$")
ax[2][1].plot(max_psi[l == 1][dr[l == 1] == 8.0E-02][:], min_lambda[l == 1][dr[l == 1] == 8.0E-02][:], "b.-", label = r"$\Delta \rho = 8/100$")

ax[2][1].set_xlabel(r"$\psi_0\,[\hbar/m]$")
ax[2][1].set_ylabel(r"$\min(\lambda)$")

fig.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()